# NN_03 — point-in-time verification

`BUILD_SPEC.md` §7, row 03: the lookahead audit, standalone, before any model. Acceptance: **building cycle *t* with and without the cycles after *t* present yields identical tensors.**

The path under audit is the one every later notebook trains on: `nn_common.bundle_cycle_tensors(bundle, transform=nn_common.normalize_cycle)` — the per-cycle rank policy verified in `NN_02` followed by the tensor construction verified in `NN_01`. The audit has three parts:

1. **Removal audit (the acceptance).** For every cycle *t* that has at least one later cycle, build *t*'s tensors from the full 14-cycle slice and again from a slice variant in which every later cycle has been removed; assert `X`, `mask`, `sector`, `quarter`, `m`, `y` and the row order are byte-identical. The headline cycle is 2022-01-01, the first cycle the baseline trained, with 8 later cycles removed.
2. **Negative control.** An audit that cannot fail proves nothing — and under the adopted policy `normalize_cycle` changes no values on this slice (`NN_02`), so the removal audit must be shown to have teeth. A deliberately wrong transform that pools cycles before ranking is run through the same audit; it must *fail*.
3. **Two independent point-in-time facts** on the same slice: the upstream pipeline's own audit record on every observation names its own cycle's cross-section size, and the walk-forward training window (`training_dataset.build_training_dataset`) excludes every quarter at or after the target and every label not yet available at the target's cutoff — the partition `NN_05` will inherit.

**No model is trained and no performance result is produced here.**

**Data mode: REAL DATA.** Everything below is computed from the cached real-data slice (`rd.slice_bundle()`, fingerprint verified in `NN_00`). The slice *variants* built here contain only real observations from that slice, with later cycles removed; nothing is added, altered, or simulated. If the raw exports are absent, every computing cell prints the explicit unavailability notice and does nothing.

Kernel: `atlasquant-venv` (= `.venv/bin/python`).

In [1]:
import nn_common as nc

rd = nc.rd
AVAILABLE = nc.provenance_banner("NN_03_pit_verification.ipynb")

neural_ranker / NN_03_pit_verification.ipynb
DATA MODE: REAL DATA.

Source: Bloomberg CSV exports under this repository's gitignored raw-data
root for multi_factor_ranking_ml -- 1,520 tickers, quarterly fundamentals
from 1980 onward, plus daily closes and macro series.

Every number below is computed from that real data through production
AtlasQuant code. No synthetic, simulated, imputed, or approximated values
are used anywhere in this notebook series.

Bounded slice: quarterly evaluation cycles 2020-10-01 .. 2024-01-01
(a genuine subset of the history, chosen so each notebook runs quickly --
not a downsample or an approximation of a longer run).

SOURCE MANIFEST (SHA-256 of every raw CSV the pipeline reads)
  raw root: /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
  fundamentals_quarterly.csv      281.3 MB  2026-08-31 23:01:10  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
  Close_Price.csv                  67.6 MB  2026-08-31 23:07:42  2

libraries         : python=3.14.5, torch=2.13.0, numpy=2.5.2, pandas=3.0.5, scipy=1.18.0, lightgbm=4.7.0


## Building a "later cycles removed" slice variant

`RealDataSlice` is a frozen dataclass, so a variant is a *new object* made with `dataclasses.replace`: it keeps only the cycles with `quarter_start <= t`, and only those cycles' feature observations and labels. It also drops the daily prices entirely and truncates the baseline result, so that nothing dated after *t* exists anywhere in the variant — if the tensor path reached for either, the audit would catch it. The variant's remaining observations are the very same objects the full slice holds.

In [2]:
from dataclasses import replace

def later_cycles_removed(bundle, t):
    kept = tuple(c for c in bundle.cycles if c.quarter_start <= t)
    keys = {c.quarter_start for c in kept}
    return replace(
        bundle,
        cycles=kept,
        feature_results={q: v for q, v in bundle.feature_results.items() if q in keys},
        labeled_quarters={q: v for q, v in bundle.labeled_quarters.items() if q in keys},
        prices_by_instrument={},
        ic_result=replace(
            bundle.ic_result,
            cycle_results=tuple(r for r in bundle.ic_result.cycle_results if r.cycle.quarter_start in keys),
        ),
    )

TENSOR_FIELDS = ("X", "mask", "sector", "quarter", "m", "y")

def byte_identical(a, b):
    # numpy bytes for every tensor (NaN positions count), plus row order and label timing
    return (
        all(getattr(a, k).tobytes() == getattr(b, k).tobytes() for k in TENSOR_FIELDS)
        and a.instrument_ids == b.instrument_ids
        and a.label_available_at == b.label_available_at
    )

## Headline: cycle 2022-01-01 with and without the 8 later cycles

The first cycle the baseline trained on. `full` is built from the 14-cycle slice; `trunc` from the variant that ends at 2022-01-01.

In [3]:
if rd.require_data():
    import numpy as np

    bundle = rd.slice_bundle()
    T = bundle.cycles[5].quarter_start
    assert T.isoformat() == "2022-01-01"

    FULL = nc.bundle_cycle_tensors(bundle, transform=nc.normalize_cycle)
    variant = later_cycles_removed(bundle, T)
    TRUNC = nc.bundle_cycle_tensors(variant, transform=nc.normalize_cycle)

    full, trunc = FULL[T], TRUNC[T]
    print("cycle                 ", T, "cutoff", bundle.cycles[5].cutoff)
    print("cycles in full slice  ", len(bundle.cycles), f"({bundle.cycles[0].quarter_start} .. {bundle.cycles[-1].quarter_start})")
    print("cycles in variant     ", len(variant.cycles), f"({variant.cycles[0].quarter_start} .. {variant.cycles[-1].quarter_start})")
    print("later cycles removed  ", len(bundle.cycles) - len(variant.cycles))
    print("variant feature_results / labeled_quarters / prices / baseline cycles:",
          len(variant.feature_results), "/", len(variant.labeled_quarters), "/", len(variant.prices_by_instrument), "/", len(variant.ic_result.cycle_results))
    print("N                     ", full.n, "/", trunc.n)
    print()
    print(f"{'tensor':<10}{'shape':<14}{'byte-identical':>15}")
    for k in TENSOR_FIELDS:
        a, b = getattr(full, k), getattr(trunc, k)
        print(f"{k:<10}{str(a.shape):<14}{str(a.tobytes() == b.tobytes()):>15}")
    print(f"{'row order':<10}{str((full.n,)):<14}{str(full.instrument_ids == trunc.instrument_ids):>15}")
    print(f"{'label_available_at':<24}{str(full.label_available_at == trunc.label_available_at):>1}   ({full.label_available_at})")
    assert byte_identical(full, trunc)
    assert max(variant.cycles, key=lambda c: c.quarter_start).quarter_start == T
    assert all(q <= T for q in variant.feature_results) and all(q <= T for q in variant.labeled_quarters)

cycle                  2022-01-01 cutoff 2021-12-31
cycles in full slice   14 (2020-10-01 .. 2024-01-01)
cycles in variant      6 (2020-10-01 .. 2022-01-01)
later cycles removed   8
variant feature_results / labeled_quarters / prices / baseline cycles: 6 / 6 / 0 / 6
N                      1468 / 1468

tensor    shape          byte-identical
X         (1468, 63)               True
mask      (1468, 63)               True
sector    (1468,)                  True
quarter   (1468,)                  True
m         (6,)                     True
y         (1468,)                  True
row order (1468,)                  True
label_available_at      True   (2022-04-01 00:00:00)


## Every cycle that has a later cycle

The same audit for all 13 such cycles, each against its own variant (1 to 13 later cycles removed).

In [4]:
if rd.require_data():
    print(f"{'quarter_start':<14}{'N':>6}{'kept':>6}{'removed':>9}{'X':>7}{'mask':>7}{'sector':>8}{'quarter':>9}{'m':>6}{'y':>6}{'order':>7}{'all':>6}")
    audited = 0
    for cycle in bundle.cycles[:-1]:
        t = cycle.quarter_start
        v = later_cycles_removed(bundle, t)
        tr = nc.bundle_cycle_tensors(v, transform=nc.normalize_cycle)[t]
        fu = FULL[t]
        flags = {k: getattr(fu, k).tobytes() == getattr(tr, k).tobytes() for k in TENSOR_FIELDS}
        flags["order"] = fu.instrument_ids == tr.instrument_ids
        ok = byte_identical(fu, tr)
        print(
            f"{t.isoformat():<14}{fu.n:>6}{len(v.cycles):>6}{len(bundle.cycles) - len(v.cycles):>9}"
            + "".join(f"{str(flags[k]):>{w}}" for k, w in (("X", 7), ("mask", 7), ("sector", 8), ("quarter", 9), ("m", 6), ("y", 6), ("order", 7)))
            + f"{str(ok):>6}"
        )
        assert ok
        audited += 1
    print(f"\ncycles audited: {audited}; byte-identical with later cycles removed: {audited}")
    assert audited == 13

quarter_start      N  kept  removed      X   mask  sector  quarter     m     y  order   all
2020-10-01      1423     1       13   True   True    True     True  True  True   True  True
2021-01-01      1430     2       12   True   True    True     True  True  True   True  True


2021-04-01      1436     3       11   True   True    True     True  True  True   True  True
2021-07-01      1451     4       10   True   True    True     True  True  True   True  True


2021-10-01      1462     5        9   True   True    True     True  True  True   True  True


2022-01-01      1468     6        8   True   True    True     True  True  True   True  True


2022-04-01      1469     7        7   True   True    True     True  True  True   True  True


...

2022-10-01      1474     9        5   True   True    True     True  True  True   True  True


2023-01-01      1478    10        4   True   True    True     True  True  True   True  True


2023-04-01      1479    11        3   True   True    True     True  True  True   True  True


2023-07-01      1488    12        2   True   True    True     True  True  True   True  True


2023-10-01      1493    13        1   True   True    True     True  True  True   True  True

cycles audited: 13; byte-identical with later cycles removed: 13


## Negative control: the audit must be able to fail

A deliberately **wrong** transform, used only here and only to show the removal audit has power: it ranks the five policy features over the cross-sections of *all* cycles in the slice pooled together, then splits the rows back into cycles — exactly the pooling `BUILD_SPEC.md` §6 forbids and `cross_sectional.py`'s docstring warns never to do. It is not the policy, it is not in `nn_common.py`, and nothing downstream uses it. Run through the same audit on 2022-01-01, it must produce *different* tensors when later cycles are removed, because the later cycles' values then participate in the ranks.

In [5]:
if rd.require_data():
    from collections import defaultdict

    def pooled_rank_FOIL(slice_variant):
        # NEGATIVE CONTROL ONLY -- deliberately violates the single-cycle scope.
        ordered = [(c.quarter_start, o) for c in slice_variant.cycles for o in slice_variant.feature_results[c.quarter_start].observations]
        pooled = nc.apply_cross_sectional_rank_normalization([o for _, o in ordered], nc.RANK_NORMALIZED)
        by_cycle = defaultdict(list)
        for (q, _), o in zip(ordered, pooled):
            by_cycle[q].append(o)
        return {
            c.quarter_start: nc.build_cycle_tensors(c, by_cycle[c.quarter_start], slice_variant.labeled_quarters.get(c.quarter_start))
            for c in slice_variant.cycles
        }

    foil_full = pooled_rank_FOIL(bundle)[T]
    foil_trunc = pooled_rank_FOIL(variant)[T]
    ranked_idx = [nc.STOCK_CONTINUOUS.index(n) for n in nc.RANK_NORMALIZED]
    a, b = foil_full.X[:, ranked_idx], foil_trunc.X[:, ranked_idx]
    ok = ~np.isnan(a)
    differing = int((a[ok] != b[ok]).sum())
    print("pooled foil on 2022-01-01, full slice vs later cycles removed:")
    print("  X byte-identical            ", a.tobytes() == b.tobytes())
    print("  differing entries           ", differing, "of", int(ok.sum()), "non-missing entries in the five ranked columns")
    print("  max |difference|            ", f"{float(np.max(np.abs(a[ok] - b[ok]))):.3g}")
    print("  vs the correct per-cycle X  ", int((a[ok] != full.X[:, ranked_idx][ok]).sum()), "entries differ from the audited path's values")
    print("  pass-through columns        ", "byte-identical" if foil_full.X[:, [j for j in range(63) if j not in ranked_idx]].tobytes() == foil_trunc.X[:, [j for j in range(63) if j not in ranked_idx]].tobytes() else "differ")
    assert not byte_identical(foil_full, foil_trunc), "the removal audit failed to detect deliberate cross-cycle pooling"
    assert differing > 0
    print("\nthe removal audit detects cross-cycle pooling: True")

pooled foil on 2022-01-01, full slice vs later cycles removed:
  X byte-identical             False
  differing entries            7204 of 7207 non-missing entries in the five ranked columns
  max |difference|             0.000426
  vs the correct per-cycle X   7202 entries differ from the audited path's values
  pass-through columns         byte-identical

the removal audit detects cross-cycle pooling: True


## The upstream pipeline's own scope record, every cycle

The five policy features were ranked *upstream*, by the production feature pipeline, when the slice was built. The production function stamps every observation it touches with the size of the cross-section it ranked over. If any cycle had been ranked against a pooled panel, that size would exceed the cycle's own N.

In [6]:
if rd.require_data():
    print(f"{'quarter_start':<14}{'N':>6}{'rank records/obs':>18}{'recorded size':>15}{'features':>10}{'== own N':>10}")
    for cycle in bundle.cycles:
        obs = bundle.feature_results[cycle.quarter_start].observations
        records = [[r for r in o.audit_trail.records if r.stage == "cross_sectional_rank_normalization"] for o in obs]
        per_obs = {len(r) for r in records}
        sizes = {r[0].data["cross_section_size"] for r in records}
        feats = {tuple(r[0].data["normalized_features"]) for r in records}
        ok = per_obs == {1} and sizes == {len(obs)} and feats == {nc.RANK_NORMALIZED}
        print(f"{cycle.quarter_start.isoformat():<14}{len(obs):>6}{str(sorted(per_obs)):>18}{str(sorted(sizes)):>15}{len(next(iter(feats))):>10}{str(ok):>10}")
        assert ok
    print("\nevery observation in the slice was ranked upstream within its own cycle's cross-section only: True")

quarter_start      N  rank records/obs  recorded size  features  == own N
2020-10-01      1423               [1]         [1423]         5      True
2021-01-01      1430               [1]         [1430]         5      True
2021-04-01      1436               [1]         [1436]         5      True
2021-07-01      1451               [1]         [1451]         5      True
2021-10-01      1462               [1]         [1462]         5      True
2022-01-01      1468               [1]         [1468]         5      True
2022-04-01      1469               [1]         [1469]         5      True
2022-07-01      1472               [1]         [1472]         5      True
2022-10-01      1474               [1]         [1474]         5      True
2023-01-01      1478               [1]         [1478]         5      True
2023-04-01      1479               [1]         [1479]         5      True
2023-07-01      1488               [1]         [1488]         5      True
2023-10-01      1493               [1]

## The walk-forward partition `NN_05` will inherit

`training_dataset.build_training_dataset` (reused, not reimplemented) builds the trailing window for a target cycle from the labeled quarters. Two rules are asserted on every cycle: every included quarter is strictly before the target, and every included row's `label_available_at` is strictly before the target's training cutoff. The immediately preceding quarter is always excluded — its labels are realized at the target's own entry date, which is after the cutoff — so a model for cycle *t* trains on at most the quarters up to *t − 2*. The resulting eligibility must match the baseline's recorded training state cycle for cycle.

In [7]:
if rd.require_data():
    from datetime import datetime, time as _time

    from atlas_quant.strategies.multi_factor_ranking_ml.config import FEATURE_SCHEMA_VERSION
    from atlas_quant.strategies.multi_factor_ranking_ml.model_training import TrainingState
    from atlas_quant.strategies.multi_factor_ranking_ml.training_dataset import (
        build_training_dataset,
        check_training_eligibility,
    )

    cfg = rd.config()
    baseline_state = {r.cycle.quarter_start: r.training_state for r in bundle.ic_result.cycle_results}
    print(f"{'target':<14}{'training_cutoff':<18}{'quarters':>9}{'first':>12}{'last':>12}{'rows':>7}{'eligible':>9}  {'baseline state':<32}{'excluded: not-yet-available / target-or-later'}")
    windows = {}
    for cycle in bundle.cycles:
        training_cutoff = datetime.combine(cycle.cutoff, _time.min)
        dataset = build_training_dataset(
            cycle.quarter_start, training_cutoff, bundle.labeled_quarters,
            strategy_id=cfg.strategy_id, feature_schema_version=FEATURE_SCHEMA_VERSION,
            ml_train_years=cfg.ml_train_years, model_config_identity=cfg.model.identity(),
        )
        eligibility = check_training_eligibility(dataset, min_train_quarters=cfg.min_train_quarters)
        assert all(q < cycle.quarter_start for q in dataset.included_quarters)
        assert all(ts < training_cutoff for ts in dataset.label_available_timestamps)
        assert sum(dataset.groups) == dataset.total_row_count
        reasons = {}
        for e in dataset.excluded_quarters:
            reasons[e.reason] = reasons.get(e.reason, 0) + 1
        not_yet = reasons.get("label not yet available by training_cutoff", 0)
        later = reasons.get("target quarter or later", 0)
        state = baseline_state[cycle.quarter_start]
        assert eligibility.eligible == (state == TrainingState.TRAINED)
        windows[cycle.quarter_start] = dataset.included_quarters
        first = dataset.included_quarters[0].isoformat() if dataset.included_quarters else "-"
        last = dataset.included_quarters[-1].isoformat() if dataset.included_quarters else "-"
        print(
            f"{cycle.quarter_start.isoformat():<14}{training_cutoff.date().isoformat():<18}{len(dataset.included_quarters):>9}{first:>12}{last:>12}"
            f"{dataset.total_row_count:>7}{str(eligibility.eligible):>9}  {state.value:<32}{not_yet} / {later}"
        )
    trained = [q for q, w in windows.items() if baseline_state[q] == TrainingState.TRAINED]
    print(f"\ntrained cycles: {len(trained)}; training quarters per trained cycle: {min(len(windows[q]) for q in trained)} .. {max(len(windows[q]) for q in trained)}")
    print("every included quarter < target and every included label available before the training cutoff: True")
    print("eligibility matches the baseline's recorded training state on all 14 cycles: True")

target        training_cutoff    quarters       first        last   rows eligible  baseline state                  excluded: not-yet-available / target-or-later
2020-10-01    2020-09-30                0           -           -      0    False  skipped_insufficient_quarters   0 / 13
2021-01-01    2020-12-31                0           -           -      0    False  skipped_insufficient_quarters   1 / 12
2021-04-01    2021-03-31                1  2020-10-01  2020-10-01   1423    False  skipped_insufficient_quarters   1 / 11
2021-07-01    2021-06-30                2  2020-10-01  2021-01-01   2853    False  skipped_insufficient_quarters   1 / 10
2021-10-01    2021-09-30                3  2020-10-01  2021-04-01   4289    False  skipped_insufficient_quarters   1 / 9
2022-01-01    2021-12-31                4  2020-10-01  2021-07-01   5740     True  trained                         1 / 8
2022-04-01    2022-03-31                5  2020-10-01  2021-10-01   7202     True  trained                   

2022-07-01    2022-06-30                6  2020-10-01  2022-01-01   8670     True  trained                         1 / 6

2022-10-01    2022-09-30                7  2020-10-01  2022-04-01  10139     True  trained                         1 / 5


2023-01-01    2022-12-31                8  2020-10-01  2022-07-01  11611     True  trained                         1 / 4
2023-04-01    2023-03-31                9  2020-10-01  2022-10-01  13085     True  trained                         1 / 3


2023-07-01    2023-06-30               10  2020-10-01  2023-01-01  14563     True  trained                         1 / 2
2023-10-01    2023-09-30               11  2020-10-01  2023-04-01  16042     True  trained                         1 / 1


2024-01-01    2023-12-31               12  2020-10-01  2023-07-01  17530     True  trained                         1 / 0

trained cycles: 9; training quarters per trained cycle: 4 .. 12
every included quarter < target and every included label available before the training cutoff: True
eligibility matches the baseline's recorded training state on all 14 cycles: True


## Point-in-time statement

Stated explicitly, per `CLAUDE.md` and `BUILD_SPEC.md` §6, and checked above rather than assumed:

- **Normalization uses only the selected cycle's own cross-section, and no future cycle can affect it.** For all 13 cycles that have a later cycle, removing every later cycle — its observations, its labels, all daily prices, and the baseline result — leaves `X`, `mask`, `sector`, `quarter`, `m`, `y`, the row order and the label timestamp byte-identical. The audit is not vacuous: a transform that pools cycles before ranking fails it on the same cycle.
- **The upstream pipeline obeyed the same scope**: every observation's production rank record names its own cycle's N.
- **The training window sees only the past**: every included quarter precedes the target and every included label was available before the target's cutoff, and the eligibility that rule produces reproduces the baseline's training states exactly.
- What this notebook does *not* claim: anything about the early-stopping holdout (which `NN_04`/`NN_05` must draw from *inside* the training window, never at or after the target) or about any temporal lookback (Stage D, out of scope). Those are checked where they are introduced.

## What this established

Every number below was printed by a cell above in this executed run. **No model was trained and no performance result was produced.**

**The acceptance holds.** For cycle 2022-01-01 (N = 1,468; cutoff 2021-12-31; the first cycle the baseline trained), tensors built from the full 14-cycle slice and from a variant holding only the 6 cycles up to 2022-01-01 — 8 later cycles removed, all daily prices dropped, baseline result truncated to 6 cycles — are byte-identical in `X (1468, 63)`, `mask (1468, 63)`, `sector (1468,)`, `quarter (1468,)`, `m (6,)`, `y (1468,)`, the row order, and the label timestamp (2022-04-01). The same audit passed for all 13 cycles that have a later cycle, with 1 to 13 later cycles removed.

**Normalization uses only the selected cycle's own cross-section, and no future cycle can affect it.** That is the direct reading of the 13-for-13 result on the exact path every later notebook trains on (`bundle_cycle_tensors` with `normalize_cycle`).

**The audit has power.** A deliberately wrong transform that pools all cycles before ranking, run through the same audit on 2022-01-01, produced tensors that differ in 7,204 of the 7,207 non-missing entries of the five ranked columns (max |difference| 0.000426; 7,202 entries differ from the correctly built values), while its pass-through columns stayed byte-identical. The removal audit detects cross-cycle pooling; the 13 passes above are not vacuous.

**The upstream pipeline obeyed the same scope.** On all 14 cycles, every observation carries exactly one production rank record, naming the five policy features and a cross-section size equal to its own cycle's N (1,423 .. 1,494).

**The walk-forward partition `NN_05` inherits, verified with `build_training_dataset`.** On every one of the 14 cycles, every included quarter is strictly before the target and every included label was available strictly before the target's training cutoff; the immediately preceding quarter is excluded every time (its labels are realized on the target's own entry date). 9 cycles are eligible, 2022-01-01 .. 2024-01-01, training on 4 to 12 quarters (5,740 to 17,530 rows); the eligibility reproduces the baseline's recorded training state on all 14 cycles. Consequently the FiLM conditioner in `NN_04`–`NN_06` will be fit on between 4 and 12 distinct macro vectors per cycle.

**Not claimed here:** the early-stopping holdout (must come from inside the training window; checked in `NN_04`/`NN_05` where it is introduced) and any temporal lookback (Stage D, out of scope).

**What `NN_04` needs from here.** The tensors are point-in-time clean and fixed; `NN_04` may build the encoder, embeddings, FiLM and ListNet loss on a single cycle for the capacity check, scaling the 58 pass-through columns of `X` with fixed, non-fitted elementwise maps only, zero-filling under the mask at input time, and reporting the overfit-one-quarter result as a capacity check, not a result.